## ch14 Do you have enough data
- https://github.com/mattharrison/effective_xgboost_book/blob/main/xgbcode.ipynb

<div style="text-align: right"> <b>Author : Kwang Myung Yu</b></div>
<div style="text-align: right"> Initial upload: 2023.8.4</div>
<div style="text-align: right"> Last update: 2023.8.4</div>

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
#plt.style.use('ggplot')
plt.style.use('seaborn-whitegrid')
%matplotlib inline

??? ?????? ?? ??.

In [ ]:
import dtreeviz
from feature_engine import encoding, imputation
import numpy as np
import pandas as pd
from sklearn import base, compose, datasets, ensemble, \
    metrics, model_selection, pipeline, preprocessing, tree
import scikitplot
import xgboost as xgb
import yellowbrick.model_selection as ms
from yellowbrick import classifier

import urllib
import zipfile

import xg_helpers as xhelp
from xg_helpers import my_dot_export

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/'\
'kaggle-survey-2018.zip'
fname = 'kaggle-survey-2018.zip'
member_name = 'multipleChoiceResponses.csv'

In [ ]:
raw = xhelp.extract_zip(url, fname, member_name)
## Create raw X and raw y
kag_X, kag_y = xhelp.get_rawX_y(raw, 'Q6')

In [ ]:
## Split data
kag_X_train, kag_X_test, kag_y_train, kag_y_test = \
model_selection.train_test_split(
kag_X, kag_y, test_size=.3, random_state=42, stratify=kag_y)

In [ ]:
## Transform X with pipeline
X_train = xhelp.kag_pl.fit_transform(kag_X_train)
X_test = xhelp.kag_pl.transform(kag_X_test)
## Transform y with label encoder
label_encoder = preprocessing.LabelEncoder()
label_encoder.fit(kag_y_train)
y_train = label_encoder.transform(kag_y_train)
y_test = label_encoder.transform(kag_y_test)
# Combined Data for cross validation/etc
X = pd.concat([X_train, X_test], axis='index')
y = pd.Series([*y_train, *y_test], index=X.index)

In [ ]:
X.head()

In [ ]:
y.head()

??? ???? ??? ??? ???? ??? ????.  
????? ??? ???? ??????? ??? ?? ???? ??? ? ????. ?? ??? ??? ????.

### 14.1 Learning Curves

?? ??? ???? ??? ??? ?? ??? ??? ???? ? ??? ??? ????? ??? ????.  
???? ??? ?? ?? ??? ????? ? ???? ??? ???? ?????.  
?? ??? X?? ?? ?? ? ?? ?? ???? ?? ????, Y?? ???, ??? ?? F1 ??? ?? ??? ?? ???? ?????. 


Yellowbrick ??????? ?? ?? ??? ??? ???? ????.  
??? ??? ?? ???? ?? ?? ?? ?? ??? ?? y-??? ???? ???? ????. 

??? ??? ???? ??? ??? ??? ???? ??? ??? ?? ??? ??? ?? ??? ??? ???.

In [ ]:
params = {'learning_rate': 0.3,
'max_depth': 2,
'n_estimators': 200,
'n_jobs': -1,
'random_state': 42,
'reg_lambda': 0,
'subsample': 1}

In [ ]:
import yellowbrick.model_selection as ms

In [ ]:
fig, ax = plt.subplots(figsize = (8, 4))

fig, ax = plt.subplots(figsize=(8, 4))
viz = ms.learning_curve(xgb.XGBClassifier(**params),
X, y, ax=ax
)
ax.set_ylim(0.6, 1)

- ?? ???? ?? ???? ?? ??? ?? ??(???) ??? ???? ???? ???? ?????. ?, ?? ?? ?? ?? ???? ?? ???? ?? ??? ????? ???? ????? ?? ?????. ?? ?? ??? ??? ? ?? ?? ??? ???? ? ???? ? ?? ??? ?? ? ?? ? ????.

- ???? ?? ??? ??? ???? ?? ???? ?????? ?????? ??? ?????. ??? ??? ?? ??? 100%? ??? ???? ?? ????. ??? ??? ???? ???? ?? ?? ????. ?? ?? ?? ??? ?? ??? ??? ??? ??? ?? ????.  

- ??? ????? ??? ???? ????? ??? ??? ?? ??.

### 14.2 Learning Curves for Decision Trees

?? max_depth? 7? ?? ??? ????.

In [ ]:
# tuned tree
fig, ax = plt.subplots(figsize=(8, 4))
viz = ms.learning_curve(tree.DecisionTreeClassifier(max_depth=7),
X, y, ax=ax)
viz.ax.set_ylim(0.6, 1)

- ? ?? ???? ???? ??? ??? ? ??.

### 14.3 Underfit Learning Curves

In [ ]:
# underfit
fig, ax = plt.subplots(figsize=(8, 4))
viz = ms.learning_curve(tree.DecisionTreeClassifier(max_depth=1),
X, y, ax=ax
)
ax.set_ylim(0.6, 1)

### 14.4 Overfit Learning Curves

In [ ]:
# overfit
fig, ax = plt.subplots(figsize=(8, 4))
viz = ms.learning_curve(tree.DecisionTreeClassifier(),
      X, y, ax=ax
)
ax.set_ylim(0.6, 1)